# TCGA-LGG Brain MRI Glioma Segmentation - Starter Notebook

Welcome to the **TCGA-LGG Brain MRI Glioma Segmentation Challenge**!

### Challenge Goals:
1. **Segment** hyperintense FLAIR abnormalities in multi-channel (pre-contrast, FLAIR, post-contrast) brain MRI slices.
2. **Respect Strict Patient-Level Splitting** to eliminate data leakage across patient scans.
3. **Evaluate** model predictions using the official **Dice Similarity Coefficient (DSC)**, **IoU**, and **95th percentile Hausdorff Distance (HD95)**.

This tutorial guides you from exploratory MRI scan visualization to building and training a U-Net, generating prediction masks, and computing official leaderboard benchmark scores.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch.utils.data import DataLoader

# Add src and evaluation to path
sys.path.append(str(Path.cwd().parent / "src"))
sys.path.append(str(Path.cwd().parent / "evaluation"))

from dataset import TCGALGGDataset
from model import UNet
from losses import DiceBCELoss
from utils import visualize_mri_slice, overlay_mask
from metrics import compute_dice_score, compute_iou_score, compute_segmentation_metrics
from validate_submission import validate_submission_masks

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

## 1. Prepare Mock or Official Dataset
Generate mock data if you want to test the entire pipeline locally without downloading 3GB of MRI scans:

In [ ]:
data_script = Path.cwd().parent / "data" / "download.py"
mock_dir = Path.cwd().parent / "data" / "mock_lgg"

if not (mock_dir / "train_manifest.csv").exists():
    print("Generating mock LGG dataset...")
    !python "{data_script}" --generate-mock --output-dir "{mock_dir}"
else:
    print(f"Dataset found at {mock_dir}")

## 2. Exploratory Data Analysis & MRI Visualization
Let's inspect training slices and visualize the FLAIR abnormality tumor mask overlaid onto the brain MRI scan.

In [ ]:
train_ds = TCGALGGDataset(
    manifest_csv=str(mock_dir / "train_manifest.csv"),
    images_dir=str(mock_dir / "images"),
    masks_dir=str(mock_dir / "masks")
)
print(f"Loaded {len(train_ds)} slices in training split.")

# 1. Run full Exploratory Data Analysis (EDA) & Modality Dashboard from starter_code/visualize_data.py
sys.path.append(str(Path.cwd().parent / "starter_code"))
from visualize_data import create_tcga_lgg_eda_dashboard

create_tcga_lgg_eda_dashboard(
    manifest_csv=Path.cwd().parent / "data" / "public" / "train_manifest.csv",
    images_dir=mock_dir / "images",
    masks_dir=mock_dir / "masks",
    output_path=Path.cwd().parent / "starter_code" / "eda_tcga_lgg_overview.png",
    show_plot=True
)

# 2. Inspect a single tumor-bearing MRI slice
sample_idx = 0
for i in range(len(train_ds)):
    img, mask, meta = train_ds[i]
    if mask.sum() > 0:
        sample_idx = i
        break

img_t, mask_t, meta = train_ds[sample_idx]
img_np = img_t.permute(1, 2, 0).numpy()
mask_np = mask_t.squeeze(0).numpy()

print(f"Displaying Slice from Patient: {meta['patient_id']} (Slice #{meta['slice_idx']})")
visualize_mri_slice(
    image=img_np,
    mask=mask_np,
    title=f"TCGA-LGG Brain MRI - Patient {meta['patient_id']}"
)

## 3. Instantiate U-Net Architecture
We use a classic symmetric U-Net with DoubleConv encoder stages, bottleneck dropout, and skip connections.

In [ ]:
model = UNet(in_channels=3, num_classes=1, init_features=32, use_batchnorm=True)
model.to(device)
print("U-Net initialized with:", sum(p.numel() for p in model.parameters() if p.requires_grad), "trainable parameters.")

## 4. Run Training Step with Composite Dice + BCE Loss

In [ ]:
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
criterion = DiceBCELoss(dice_weight=0.6, bce_weight=0.4)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

model.train()
for step, (images, masks, _) in enumerate(train_loader):
    images = images.to(device)
    masks = masks.to(device)
    
    optimizer.zero_grad()
    logits = model(images)
    loss = criterion(logits, masks)
    loss.backward()
    optimizer.step()
    
    if (step + 1) % 4 == 0:
        print(f"Step {step + 1}/{len(train_loader)} - Batch DiceBCE Loss: {loss.item():.4f}")
    if step >= 8:  # Demo stop
        break

## 5. Generate Test Predictions & Save Submission Masks
We evaluate test slices and export binary mask images into the submission directory.

In [ ]:
val_ds = TCGALGGDataset(
    manifest_csv=str(mock_dir / "val_manifest.csv"),
    images_dir=str(mock_dir / "images"),
    masks_dir=str(mock_dir / "masks")
)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)

pred_dir = Path.cwd().parent / "sample_submission" / "predictions"
pred_dir.mkdir(parents=True, exist_ok=True)

model.eval()
with torch.no_grad():
    for images, _, metas in val_loader:
        images = images.to(device)
        logits = model(images)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).cpu().numpy().squeeze(1)
        
        for i in range(len(preds)):
            mask_img = (preds[i] * 255).astype(np.uint8)
            out_file = pred_dir / metas["mask_file"][i]
            Image.fromarray(mask_img).save(out_file)

print(f"Saved {len(list(pred_dir.glob('*.png')))} prediction masks to {pred_dir}")

## 6. Official Submission Evaluation Scoreboard

In [ ]:
gt_val_dir = mock_dir / "masks_val"
results = compute_segmentation_metrics(gt_val_dir, pred_dir)

print("=" * 60)
print("             TCGA-LGG BENCHMARK RESULTS                    ")
print("=" * 60)
print(f"Mean Slice Dice:              {results['mean_slice_dice']:.4f}")
print(f"Patient 3D Volumetric Dice:   {results['mean_patient_3d_dice']:.4f}")
print(f"Mean Intersection over Union: {results['mean_iou']:.4f}")
print(f"95th Percentile Hausdorff:    {results['mean_hd95_pixels']:.2f} pixels")
print(f"Sensitivity (Recall):         {results['mean_sensitivity']:.4f}")
print(f"Specificity:                  {results['mean_specificity']:.4f}")
print("=" * 60)